# Tests helper-level para OP-08 / OP-09

## Bloque 0. Preparación

### 0.1 Imports generales

In [1]:
import copy
import json
import re
import shutil
import tempfile
from pathlib import Path

import pandas as pd
import h3

### 0.2 Imports del módulo

In [2]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.datasets import TripDataset, FlowDataset
from pylondrina.reports import FlowBuildReport, Issue, OperationReport
from pylondrina.errors import ValidationError, SchemaError, ExportError

from pylondrina.transforms.flows import (
    FlowBuildOptions,
    _resolve_and_precheck_build_request,
    _prepare_buildable_movements,
    _aggregate_flows,
    _build_flow_to_trips,
    _build_flow_dataset,
    _build_flow_report_and_event,
    _normalize_h3_series,
    _infer_h3_resolution_from_columns,
    _make_window_start,
    _make_window_end,
    _extract_validated_flag,
    _extract_temporal_tier,
)
from pylondrina.export.flows import (
    ExportFlowsOptions,
    FlowExportResult,
    _resolve_export_request,
    _preflight_export_flows,
    _build_flowmap_tables,
    _materialize_flowmap_export,
    _append_export_event_or_warning,
    _sanitize_folder_name,
    _generate_folder_name,
    _series_is_csv_serializable,
)

### 0.3 Helpers de apoyo para test

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}")


### 0.4 Configuración visual y directorio visible

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

HELPER_ROOT = Path("tmp_helper_tests")
if HELPER_ROOT.exists():
    shutil.rmtree(HELPER_ROOT)
HELPER_ROOT.mkdir(parents=True, exist_ok=True)

print("HELPER_ROOT =", HELPER_ROOT.resolve())

HELPER_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\build_flows\tmp_helper_tests


## Bloque 1. Fixtures reutilizables mínimas

### 1.1 Factories de schema y datasets

In [5]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema_minimal_for_flows() -> TripSchema:
    fields = {
        "movement_id": make_field("movement_id", "string", required=True),
        "origin_h3_index": make_field("origin_h3_index", "string", required=True),
        "destination_h3_index": make_field("destination_h3_index", "string", required=True),
        "origin_time_utc": make_field("origin_time_utc", "datetime", required=False),
        "destination_time_utc": make_field("destination_time_utc", "datetime", required=False),
        "mode": make_field(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(values=["bus", "metro", "walk"], extendable=True),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "other"], extendable=True),
        ),
        "user_gender": make_field(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(values=["M", "F"], extendable=True),
        ),
        "trip_weight": make_field("trip_weight", "float", required=False),
    }
    return TripSchema(
        version="1.1",
        fields=fields,
        required=["movement_id", "origin_h3_index", "destination_h3_index"],
    )


def make_trip_schema_effective_for_flows() -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective={
            "movement_id": "string",
            "origin_h3_index": "string",
            "destination_h3_index": "string",
            "origin_time_utc": "datetime",
            "destination_time_utc": "datetime",
            "mode": "categorical",
            "purpose": "categorical",
            "user_gender": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": ["bus", "metro", "walk"],
            "purpose": ["work", "study", "other"],
            "user_gender": ["M", "F"],
        },
        temporal={"tier": "tier_1"},
        fields_effective=[
            "movement_id",
            "origin_h3_index",
            "destination_h3_index",
            "origin_time_utc",
            "destination_time_utc",
            "mode",
            "purpose",
            "user_gender",
            "trip_weight",
        ],
    )


def make_buildable_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m0", "m1", "m2", "m3"],
            "origin_h3_index": [
                "8828308281fffff",
                "8828308281fffff",
                "8828308281fffff",
                "882830828dfffff",
            ],
            "destination_h3_index": [
                "8828308285fffff",
                "8828308285fffff",
                "8828308285fffff",
                "8828308287fffff",
            ],
            "origin_time_utc": pd.to_datetime(
                [
                    "2026-04-01T08:10:00Z",
                    "2026-04-01T08:20:00Z",
                    "2026-04-01T08:45:00Z",
                    "2026-04-01T09:10:00Z",
                ],
                utc=True,
            ),
            "destination_time_utc": pd.to_datetime(
                [
                    "2026-04-01T08:35:00Z",
                    "2026-04-01T08:50:00Z",
                    "2026-04-01T09:05:00Z",
                    "2026-04-01T09:40:00Z",
                ],
                utc=True,
            ),
            "mode": ["bus", "bus", "metro", "bus"],
            "purpose": ["work", "work", "study", "work"],
            "user_gender": ["M", "F", "F", "M"],
            "trip_weight": [1.0, 2.0, 1.5, 3.0],
        }
    )


def make_tripdataset_for_flows(
    *,
    validated: bool = True,
    tier: str = "tier_1",
    include_dataset_id: bool = True,
) -> TripDataset:
    metadata = {
        "is_validated": validated,
        "events": [
            {
                "op": "import_trips",
                "ts_utc": "2026-04-01T00:00:00Z",
                "summary": {"rows_out": 4},
            },
            {
                "op": "validate_trips",
                "ts_utc": "2026-04-01T00:10:00Z",
                "summary": {"ok": validated},
            },
        ],
        "temporal": {
            "tier": tier,
            "fields_present": ["origin_time_utc", "destination_time_utc"],
        },
        "h3": {"resolution": 8},
    }
    if include_dataset_id:
        metadata["dataset_id"] = "trips_case_001"

    return TripDataset(
        data=make_buildable_trip_df(),
        schema=make_trip_schema_minimal_for_flows(),
        schema_version="1.1",
        provenance={"source_name": "demo_source", "source": {"name": "demo_source"}},
        metadata=metadata,
        schema_effective=make_trip_schema_effective_for_flows(),
    )


def make_flowdataset_for_export(
    *,
    include_dataset_id: bool = True,
    with_extra_fields: bool = True,
) -> FlowDataset:
    flows_df = pd.DataFrame(
        {
            "flow_id": ["flow_0000000", "flow_0000001"],
            "origin_h3_index": ["8828308281fffff", "882830828dfffff"],
            "destination_h3_index": ["8828308285fffff", "8828308287fffff"],
            "flow_count": [2, 1],
            "flow_value": [3.0, 3.0],
            "window_start_utc": pd.to_datetime(
                ["2026-04-01T08:00:00Z", "2026-04-01T09:00:00Z"], utc=True
            ),
            "window_end_utc": pd.to_datetime(
                ["2026-04-01T09:00:00Z", "2026-04-01T10:00:00Z"], utc=True
            ),
            "mode": ["bus", "metro"],
            "purpose": ["work", "study"],
            "user_gender": ["M", "F"],
        }
    )
    if not with_extra_fields:
        flows_df = flows_df[["flow_id", "origin_h3_index", "destination_h3_index", "flow_count", "flow_value"]].copy()

    metadata = {
        "events": [],
        "is_validated": False,
    }
    if include_dataset_id:
        metadata["dataset_id"] = "flows_case_001"

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=pd.DataFrame(
            {
                "flow_id": ["flow_0000000", "flow_0000000", "flow_0000001"],
                "movement_id": ["m0", "m1", "m3"],
            }
        ),
        aggregation_spec={
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
            "keep_flow_to_trips": True,
            "require_validated": True,
            "strict": False,
            "max_issues": 1000,
            "effective_flow_keys": ["origin_h3_index", "destination_h3_index", "window_start_utc", "window_end_utc", "mode"],
        },
        source_trips=None,
        metadata=metadata,
        provenance={
            "source_name": "demo_source",
            "derived_from": [{"source_type": "trips", "dataset_id": "trips_case_001"}],
        },
    )


def make_case_dir(case_name: str) -> Path:
    case_dir = HELPER_ROOT / case_name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir

## Bloque 2. Helpers internos de uso general prioritarios

### Test 2.1 - `_extract_validated_flag` y `_extract_temporal_tier`
Qué prueba: lectura robusta de metadata nueva/legacy y fallback temporal conservador.

In [6]:
assert _extract_validated_flag({"is_validated": True}) is True
assert _extract_validated_flag({"flags": {"validated": True}}) is True
assert _extract_validated_flag({}) is False
assert _extract_validated_flag(None) is False

assert _extract_temporal_tier({"temporal": {"tier": "tier_1"}}) == "tier_1"
assert _extract_temporal_tier({"temporal": {}}) == "tier_3"
assert _extract_temporal_tier({}) == "tier_3"
assert _extract_temporal_tier(None) == "tier_3"

show_ok("2.1 validated flag + temporal tier")

OK - 2.1 validated flag + temporal tier


### Test 2.2 - `_normalize_h3_series` e `_infer_h3_resolution_from_columns`
Qué prueba: normalización de H3 válidos/nulos/inválidos e inferencia de resolución sin mezcla.

In [7]:
series = pd.Series([
    "8828308281fffff",
    None,
    "not_a_real_h3",
    "882830828dfffff",
])
normalized, missing_mask, invalid_values = _normalize_h3_series(series)

assert normalized.iloc[0] == "8828308281fffff"
assert pd.isna(normalized.iloc[1])
assert pd.isna(normalized.iloc[2])
assert normalized.iloc[3] == "882830828dfffff"
assert missing_mask.tolist() == [False, True, False, False]
assert invalid_values == ["not_a_real_h3"]

res_inferred, mixed = _infer_h3_resolution_from_columns(
    pd.Series(["8828308281fffff"]),
    pd.Series(["882830828dfffff"]),
)
assert res_inferred == 8
assert mixed is False

show_ok("2.2 normalize h3 + infer resolution")

OK - 2.2 normalize h3 + infer resolution


### Test 2.3 - `_make_window_start` y `_make_window_end`
Qué prueba: binning temporal para `hour`, `day` y `week`.

In [10]:
series_utc = pd.to_datetime(
    ["2026-04-01T08:17:00Z", "2026-04-02T15:42:00Z"], utc=True
).tz_convert(None)

hour_start = _make_window_start(series_utc, "hour")
hour_end = _make_window_end(hour_start, "hour")
assert str(hour_start.iloc[0]) == "2026-04-01 08:00:00"
assert str(hour_end.iloc[0]) == "2026-04-01 09:00:00"

day_start = _make_window_start(series_utc, "day")
day_end = _make_window_end(day_start, "day")
assert str(day_start.iloc[0]) == "2026-04-01 00:00:00"
assert str(day_end.iloc[0]) == "2026-04-02 00:00:00"

week_start = _make_window_start(series_utc, "week")
week_end = _make_window_end(week_start, "week")
assert str(week_end.iloc[0] - week_start.iloc[0]) == "7 days 00:00:00"
assert str(week_start.iloc[0]) == "2026-03-30 00:00:00"
assert str(week_end.iloc[0]) == "2026-04-06 00:00:00"

display(
    pd.DataFrame(
        {
            "input": series_utc,
            "hour_start": hour_start,
            "hour_end": hour_end,
            "day_start": day_start,
            "day_end": day_end,
            "week_start": week_start,
            "week_end": week_end,
        }
    )
)
show_ok("2.3 window start/end")

,input,hour_start,hour_end,day_start,day_end,week_start,week_end
0,2026-04-01 08:17:00,2026-04-01 08:00:00,2026-04-01 09:00:00,2026-04-01,2026-04-02,2026-03-30,2026-04-06
1,2026-04-02 15:42:00,2026-04-02 15:00:00,2026-04-02 16:00:00,2026-04-02,2026-04-03,2026-03-30,2026-04-06


OK - 2.3 window start/end


### Test 2.4 - `_sanitize_folder_name`, `_generate_folder_name` y `_series_is_csv_serializable`
Qué prueba: saneamiento/generación de folder_name y detección simple de columnas no serializables a CSV.


In [11]:
flows = make_flowdataset_for_export()

assert _sanitize_folder_name("  mi folder raro  ") == "mi_folder_raro"
assert _sanitize_folder_name("***") == ""

generated = _generate_folder_name(flows)
assert "flows_flowmap" in generated
assert generated.startswith("demo_source_")

assert _series_is_csv_serializable(pd.Series(["a", 1, 2.5])) is True
assert _series_is_csv_serializable(pd.Series([[1, 2], {"a": 1}])) is False

show_ok("2.4 export general helpers")

OK - 2.4 export general helpers


## Bloque 3. Helpers principales de Build flows

### Test 3.1 - `_resolve_and_precheck_build_request` happy path

In [12]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")
options_eff, parameters = _resolve_and_precheck_build_request(
    trips,
    FlowBuildOptions(
        h3_resolution=8,
        group_by=["mode"],
        time_aggregation="hour",
        time_basis="origin",
        min_trips_per_flow=1,
        keep_flow_to_trips=True,
        require_validated=True,
        strict=False,
        max_issues=20,
    ),
)

assert isinstance(options_eff, FlowBuildOptions)
assert parameters["h3_resolution"] == 8
assert parameters["group_by"] == ["mode"]
assert parameters["time_aggregation"] == "hour"
assert parameters["keep_flow_to_trips"] is True

show_ok("3.1 resolve_and_precheck_build_request happy path")

OK - 3.1 resolve_and_precheck_build_request happy path


### Test 3.2 - `_resolve_and_precheck_build_request` fatal por trips no validados
Qué prueba: precondición principal cuando `require_validated=True`.

In [13]:
trips = make_tripdataset_for_flows(validated=False, tier="tier_1")
raised = None
try:
    _resolve_and_precheck_build_request(
        trips,
        FlowBuildOptions(require_validated=True),
    )
except ValidationError as e:
    raised = e

assert raised is not None
assert "REQUIRED_NOT_VALIDATED" in str(raised).upper()

show_ok("3.2 resolve_and_precheck_build_request fatal not validated")

OK - 3.2 resolve_and_precheck_build_request fatal not validated


### Test 3.3 - `_prepare_buildable_movements` happy path con agregación temporal
Qué prueba: working set buildable, claves efectivas y ventanas temporales.

In [15]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")
options = FlowBuildOptions(
    h3_resolution=8,
    group_by=["mode"],
    time_aggregation="hour",
    time_basis="origin",
    min_trips_per_flow=1,
    keep_flow_to_trips=False,
    require_validated=True,
)
prepared_df, issues, prep_info = _prepare_buildable_movements(trips, options)

assert len(issues) == 0
assert len(prepared_df) == 4
assert "window_start_utc" in prepared_df.columns
assert "window_end_utc" in prepared_df.columns
assert prep_info["effective_flow_keys"] == [
    "origin_h3_index",
    "destination_h3_index",
    "window_start_utc",
    "window_end_utc",
    "mode",
]
assert prep_info["n_trips_in"] == 4
assert prep_info["n_trips_eligible"] == 4
assert prep_info["n_trips_dropped"] == 0

display(trips.data)
display(prepared_df)
show_ok("3.3 prepare_buildable_movements happy path")

,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight,window_start_utc,window_end_utc
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0,2026-04-01 08:00:00,2026-04-01 09:00:00
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0,2026-04-01 08:00:00,2026-04-01 09:00:00
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5,2026-04-01 08:00:00,2026-04-01 09:00:00
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0,2026-04-01 09:00:00,2026-04-01 10:00:00


OK - 3.3 prepare_buildable_movements happy path


### Test 3.4 - `_prepare_buildable_movements` con descarte agregado por H3 faltante
Qué prueba: warning agregado y exclusión de filas no buildables.

In [17]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")
trips.data.loc[1, "destination_h3_index"] = None

prepared_df, issues, prep_info = _prepare_buildable_movements(
    trips,
    FlowBuildOptions(h3_resolution=8, group_by=None, time_aggregation="none"),
)

codes = [issue.code for issue in issues]
assert "FLOW.OUTPUT.MOVEMENTS_DROPPED_MISSING_OD_H3" in codes
assert len(prepared_df) == 3
assert prep_info["n_trips_in"] == 4
assert prep_info["n_trips_eligible"] == 3
assert prep_info["n_trips_dropped"] == 1

display(trips.data)
display(prepared_df)
show_ok("3.4 prepare_buildable_movements dropped missing OD")

,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,None,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


OK - 3.4 prepare_buildable_movements dropped missing OD


### Test 3.5 - `_aggregate_flows` con `trip_weight` y threshold
Qué prueba: cálculo de `flow_count`, `flow_value`, filtro por umbral y creación de `flow_id`.

In [18]:
prepared_df = pd.DataFrame(
    {
        "movement_id": ["m0", "m1", "m2"],
        "origin_h3_index": ["8828308281fffff", "8828308281fffff", "882830828dfffff"],
        "destination_h3_index": ["8828308285fffff", "8828308285fffff", "8828308287fffff"],
        "mode": ["bus", "bus", "metro"],
        "trip_weight": [1.0, 2.0, 5.0],
    }
)

flows_df = _aggregate_flows(
    prepared_df,
    ["origin_h3_index", "destination_h3_index", "mode"],
    FlowBuildOptions(min_trips_per_flow=2),
)

assert list(flows_df.columns) == [
    "flow_id",
    "origin_h3_index",
    "destination_h3_index",
    "mode",
    "flow_count",
    "flow_value",
]
assert len(flows_df) == 1
assert flows_df.iloc[0]["flow_count"] == 2
assert float(flows_df.iloc[0]["flow_value"]) == 3.0
assert flows_df.iloc[0]["flow_id"].startswith("flow_")

display(prepared_df)
display(flows_df)
show_ok("3.5 aggregate_flows with trip_weight + threshold")

,movement_id,origin_h3_index,destination_h3_index,mode,trip_weight
0,m0,8828308281fffff,8828308285fffff,bus,1.0
1,m1,8828308281fffff,8828308285fffff,bus,2.0
2,m2,882830828dfffff,8828308287fffff,metro,5.0


,flow_id,origin_h3_index,destination_h3_index,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,bus,2,3.0


OK - 3.5 aggregate_flows with trip_weight + threshold


### Test 3.6 - `_build_flow_to_trips` enabled
Qué prueba: backlinks mínimos `flow_id + movement_id`.

In [20]:
prepared_df = pd.DataFrame(
    {
        "movement_id": ["m0", "m1", "m2"],
        "origin_h3_index": ["8828308281fffff", "8828308281fffff", "882830828dfffff"],
        "destination_h3_index": ["8828308285fffff", "8828308285fffff", "8828308287fffff"],
        "mode": ["bus", "bus", "metro"],
    }
)
flows_df = pd.DataFrame(
    {
        "flow_id": ["flow_0000000", "flow_0000001"],
        "origin_h3_index": ["8828308281fffff", "882830828dfffff"],
        "destination_h3_index": ["8828308285fffff", "8828308287fffff"],
        "mode": ["bus", "metro"],
        "flow_count": [2, 1],
        "flow_value": [2.0, 1.0],
    }
)

flow_to_trips = _build_flow_to_trips(
    prepared_df,
    flows_df,
    FlowBuildOptions(keep_flow_to_trips=True),
)

assert list(flow_to_trips.columns) == ["flow_id", "movement_id"]
assert len(flow_to_trips) == 3
assert sorted(flow_to_trips["movement_id"].tolist()) == ["m0", "m1", "m2"]

display(prepared_df)
display(flows_df)
display(flow_to_trips)
show_ok("3.6 build_flow_to_trips")

,movement_id,origin_h3_index,destination_h3_index,mode
0,m0,8828308281fffff,8828308285fffff,bus
1,m1,8828308281fffff,8828308285fffff,bus
2,m2,882830828dfffff,8828308287fffff,metro


,flow_id,origin_h3_index,destination_h3_index,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,bus,2,2.0
1,flow_0000001,882830828dfffff,8828308287fffff,metro,1,1.0


,flow_id,movement_id
0,flow_0000000,m0
1,flow_0000000,m1
2,flow_0000001,m2


OK - 3.6 build_flow_to_trips


### Test 3.7 - `_build_flow_dataset`
Qué prueba: armado del `FlowDataset` derivado con metadata/provenance/aggregation_spec y `is_validated=False`.

In [22]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")
flows_df = pd.DataFrame(
    {
        "flow_id": ["flow_0000000"],
        "origin_h3_index": ["8828308281fffff"],
        "destination_h3_index": ["8828308285fffff"],
        "flow_count": [2],
        "flow_value": [3.0],
        "mode": ["bus"],
    }
)
flow_to_trips = pd.DataFrame({"flow_id": ["flow_0000000", "flow_0000000"], "movement_id": ["m0", "m1"]})
prep_info = {
    "effective_flow_keys": ["origin_h3_index", "destination_h3_index", "mode"],
    "n_trips_in": 4,
    "n_trips_eligible": 4,
    "n_trips_dropped": 0,
    "h3_resolution_input": 8,
}

flow_dataset = _build_flow_dataset(
    trips,
    flows_df,
    flow_to_trips,
    FlowBuildOptions(h3_resolution=8, group_by=["mode"], keep_flow_to_trips=True),
    prep_info,
)

assert isinstance(flow_dataset, FlowDataset)
assert flow_dataset.metadata["is_validated"] is False
assert flow_dataset.metadata["artifact_id"] is None
assert flow_dataset.metadata["h3"]["resolution"] == 8
assert flow_dataset.provenance["derived_from"][0]["source_type"] == "trips"
assert flow_dataset.provenance["derived_from"][0]["dataset_id"] == "trips_case_001"
assert flow_dataset.source_trips is trips
assert_json_safe(flow_dataset.aggregation_spec, "aggregation_spec")
assert_json_safe(flow_dataset.provenance, "provenance")

display(flow_dataset.flows)
display(flow_dataset.source_trips.data)
show_ok("3.7 build_flow_dataset")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode
0,flow_0000000,8828308281fffff,8828308285fffff,2,3.0,bus


,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


OK - 3.7 build_flow_dataset


### Test 3.8 - `_build_flow_report_and_event`
Qué prueba: summary mínimo estable, parameters efectivos y evento `build_flows`.

In [23]:
issues = []
prep_info = {
    "n_trips_in": 4,
    "n_trips_eligible": 4,
    "n_trips_dropped": 0,
}
flows_df = pd.DataFrame(
    {
        "flow_id": ["flow_0000000"],
        "origin_h3_index": ["8828308281fffff"],
        "destination_h3_index": ["8828308285fffff"],
        "flow_count": [2],
        "flow_value": [3.0],
    }
)
flow_to_trips = pd.DataFrame({"flow_id": ["flow_0000000", "flow_0000000"], "movement_id": ["m0", "m1"]})
options = FlowBuildOptions(
    h3_resolution=8,
    group_by=None,
    time_aggregation="none",
    time_basis="origin",
    min_trips_per_flow=1,
    keep_flow_to_trips=True,
    require_validated=True,
    strict=False,
    max_issues=1000,
)

report, event = _build_flow_report_and_event(
    issues,
    options,
    prep_info,
    flows_df,
    flow_to_trips,
)

assert isinstance(report, FlowBuildReport)
assert report.ok is True
assert report.summary == {
    "n_trips_in": 4,
    "n_trips_eligible": 4,
    "n_trips_dropped": 0,
    "n_flows_out": 1,
    "n_flow_to_trips_rows": 2,
}
assert report.parameters["h3_resolution"] == 8
assert event["op"] == "build_flows"
assert event["parameters"]["keep_flow_to_trips"] is True
assert event["summary"]["n_flows_out"] == 1
assert_json_safe(event, "build event")

show_ok("3.8 build_flow_report_and_event")

OK - 3.8 build_flow_report_and_event


## Bloque 4. Helpers principales de Export flows

### Test 4.1 - `_resolve_export_request` happy path con folder autogenerado

In [24]:
flows = make_flowdataset_for_export()
case_dir = make_case_dir("case_04_01_resolve_export_request_happy")

options_eff, export_dir, parameters = _resolve_export_request(
    flows,
    str(case_dir),
    ExportFlowsOptions(format="flowmap_blue", mode="error_if_exists", folder_name=None, extra_flow_fields=["mode"]),
)

assert isinstance(options_eff, ExportFlowsOptions)
assert options_eff.format == "flowmap_blue"
assert options_eff.mode == "error_if_exists"
assert options_eff.extra_flow_fields == ["mode"]
assert str(case_dir) in export_dir
assert parameters["format"] == "flowmap_blue"
assert parameters["folder_name"] is not None
assert parameters["_folder_name_generated"] is True

show_ok("4.1 resolve_export_request happy path")

OK - 4.1 resolve_export_request happy path


### Test 4.2 - `_resolve_export_request` fatal por colisión de directorio
Qué prueba: `mode="error_if_exists"` aborta si el directorio final ya existe.

In [25]:
flows = make_flowdataset_for_export()
case_dir = make_case_dir("case_04_02_resolve_export_request_fatal_collision")
existing = case_dir / "fixed_folder"
existing.mkdir(parents=True, exist_ok=True)

raised = None
try:
    _resolve_export_request(
        flows,
        str(case_dir),
        ExportFlowsOptions(format="flowmap_blue", mode="error_if_exists", folder_name="fixed_folder"),
    )
except ExportError as e:
    raised = e

assert raised is not None
assert "EXISTS" in str(raised).upper() or "EXPORT_DIR" in str(raised).upper()

show_ok("4.2 resolve_export_request fatal collision")

OK - 4.2 resolve_export_request fatal collision


### Test 4.3 - `_preflight_export_flows` happy path con extras válidas
Qué prueba: preflight mínimo correcto y `count_source=flow_value`.

In [26]:
flows = make_flowdataset_for_export(with_extra_fields=True)
issues, info = _preflight_export_flows(
    flows,
    ExportFlowsOptions(extra_flow_fields=["mode", "purpose", "window_start_utc"]),
)

assert len([issue for issue in issues if issue.level == "error"]) == 0
assert info["count_source"] == "flow_value"
assert info["extra_flow_fields"] == ["mode", "purpose", "window_start_utc"]

show_ok("4.3 preflight_export_flows happy path")

OK - 4.3 preflight_export_flows happy path


### Test 4.4 - `_preflight_export_flows` fatal por extra reservada
Qué prueba: `origin/dest/count` no pueden pedirse como extras.

In [27]:
flows = make_flowdataset_for_export(with_extra_fields=True)
raised = None
try:
    _preflight_export_flows(
        flows,
        ExportFlowsOptions(extra_flow_fields=["mode", "count"]),
    )
except ExportError as e:
    raised = e

assert raised is not None
assert "RESERVED" in str(raised).upper() or "EXTRA" in str(raised).upper()

show_ok("4.4 preflight_export_flows fatal reserved extra")

OK - 4.4 preflight_export_flows fatal reserved extra


### Test 4.5 - `_build_flowmap_tables`
Qué prueba: mapping externo `origin/dest/count` y construcción de `locations.csv` desde centroides H3.

In [28]:
flows = make_flowdataset_for_export(with_extra_fields=True)
flows_out_df, locations_df = _build_flowmap_tables(
    flows.flows,
    extra_flow_fields=["mode", "purpose"],
    count_source="flow_value",
)

assert list(flows_out_df.columns) == ["origin", "dest", "count", "mode", "purpose"]
assert len(flows_out_df) == 2
assert list(locations_df.columns) == ["id", "name", "lat", "lon"]
assert len(locations_df) >= 2
assert set(locations_df["id"]) == set(flows.flows["origin_h3_index"]).union(set(flows.flows["destination_h3_index"]))

show_ok("4.5 build_flowmap_tables")

OK - 4.5 build_flowmap_tables


### Test 4.6 - `_materialize_flowmap_export` happy path visible
Qué prueba: materialización completa de `flows.csv`, `locations.csv`, `metadata.json`, más `FlowExportResult` y `OperationReport` coherentes.

In [29]:
flows = make_flowdataset_for_export(with_extra_fields=True)
case_dir = make_case_dir("case_04_06_materialize_flowmap_export_happy")
export_dir = case_dir / "artifact_export"
flows_out_df, locations_df = _build_flowmap_tables(
    flows.flows,
    extra_flow_fields=["mode", "purpose"],
    count_source="flow_value",
)
parameters = {
    "output_root": str(case_dir),
    "export_dir": str(export_dir),
    "format": "flowmap_blue",
    "mode": "error_if_exists",
    "folder_name": "artifact_export",
    "extra_flow_fields": ["mode", "purpose"],
}

result, report, event_dict = _materialize_flowmap_export(
    flows,
    str(export_dir),
    flows_out_df,
    locations_df,
    parameters,
    "flow_value",
)

assert isinstance(result, FlowExportResult)
assert isinstance(report, OperationReport)
assert Path(result.export_dir).exists()
assert Path(result.artifacts["flows"]).exists()
assert Path(result.artifacts["locations"]).exists()
assert Path(result.artifacts["metadata"]).exists()
assert report.ok is True
assert report.summary["n_flows"] == 2
assert report.summary["n_locations"] == len(locations_df)
assert event_dict["op"] == "export_flows"

sidecar = json.loads(Path(result.artifacts["metadata"]).read_text(encoding="utf-8"))
assert sidecar["artifact_type"] == "flow_export"
assert sidecar["files"]["flows"] == "flows.csv"
assert sidecar["flow_dataset_ref"]["dataset_id"] == "flows_case_001"
assert sidecar["export"]["count_source"] == "flow_value"

show_ok("4.6 materialize_flowmap_export happy path")

OK - 4.6 materialize_flowmap_export happy path


### Test 4.7 - `_append_export_event_or_warning` normal
Qué prueba: append correcto del evento al pipeline propio del `FlowDataset`.

In [30]:
flows = make_flowdataset_for_export()
report = OperationReport(ok=True, issues=[], summary={"n_flows": 2}, parameters={"format": "flowmap_blue"})
event_dict = {
    "op": "export_flows",
    "ts_utc": "2026-04-01T12:00:00Z",
    "parameters": {"format": "flowmap_blue"},
    "summary": {"n_flows": 2},
}

report_out = _append_export_event_or_warning(flows, event_dict, report)

assert report_out.ok is True
assert flows.metadata["events"][-1]["op"] == "export_flows"

show_ok("4.7 append_export_event_or_warning normal")

OK - 4.7 append_export_event_or_warning normal


### Test 4.8 - `_append_export_event_or_warning` recovery con warning
Qué prueba: si falla el append del evento, no se invalida el export y queda warning agregado.

In [33]:
class BrokenEventsDict(dict):
    def __setitem__(self, key, value):
        if key == "events":
            raise RuntimeError("broken events append")
        return super().__setitem__(key, value)

flows = make_flowdataset_for_export()

base_metadata = dict(flows.metadata)
base_metadata["events"] = None
flows.metadata = BrokenEventsDict(base_metadata)

report = OperationReport(
    ok=True,
    issues=[],
    summary={"n_flows": 2},
    parameters={"format": "flowmap_blue"},
)
event_dict = {
    "op": "export_flows",
    "ts_utc": "2026-04-01T12:00:00Z",
    "parameters": {"format": "flowmap_blue"},
    "summary": {"n_flows": 2},
}

report_out = _append_export_event_or_warning(flows, event_dict, report)

codes = [issue.code for issue in report_out.issues]
assert "EXPORT_FLOWS.EVENT.APPEND_FAILED" in codes
assert report_out.ok is True

show_ok("4.8 append_export_event_or_warning recovery")

OK - 4.8 append_export_event_or_warning recovery


## Bloque 5: Smoke tests integrados de build_flows / export_flows

### 5.0 - Setup de smoke tests

Qué hace: crea un directorio visible para los smoke tests y reutiliza las fixtures ya definidas arriba.

In [36]:
from pathlib import Path
import json

from pylondrina.transforms.flows import (
    FlowBuildOptions,
    build_flows
)
from pylondrina.export.flows import (
    ExportFlowsOptions,
    FlowExportResult,
    export_flows
)

SMOKE_ROOT = make_case_dir("smoke_tests")
display({"SMOKE_ROOT": str(SMOKE_ROOT)})

{'SMOKE_ROOT': 'tmp_helper_tests\\smoke_tests'}

### Smoke 5.1 - build_flows happy path mínimo

Qué prueba: construcción básica sin segmentación ni tiempo, usando solo OD y pesos; verifica FlowDataset, FlowBuildReport, evento y metadata mínima.

In [ ]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")

options = FlowBuildOptions(
    h3_resolution=8,
    group_by=None,
    time_aggregation="none",
    min_trips_per_flow=1,
    keep_flow_to_trips=False,
    require_validated=True,
)

flow_ds, report = build_flows(trips, options=options)

assert isinstance(flow_ds, FlowDataset)
assert isinstance(report, FlowBuildReport)
assert report.ok is True

# Debe haber 2 flujos: uno agregado de 3 movements y otro de 1
assert len(flow_ds.flows) == 2
assert {"flow_id", "origin_h3_index", "destination_h3_index", "flow_count", "flow_value"}.issubset(flow_ds.flows.columns)

# flow_to_trips no se pidió
assert flow_ds.flow_to_trips is None

# Estado del dataset derivado
assert flow_ds.metadata["is_validated"] is False
assert flow_ds.metadata["events"][-1]["op"] == "build_flows"

# Provenance derivada
assert "derived_from" in flow_ds.provenance
assert isinstance(flow_ds.provenance["derived_from"], list)
assert flow_ds.source_trips is trips

display(trips.data)
display(flow_ds.flows.sort_values(["origin_h3_index", "destination_h3_index"]).reset_index(drop=True))
display(report.summary)
show_ok("5.1 build_flows happy path mínimo")

,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,3,4.5
1,flow_0000001,882830828dfffff,8828308287fffff,1,3.0


{'n_trips_in': 4,
 'n_trips_eligible': 4,
 'n_trips_dropped': 0,
 'n_flows_out': 2,
 'n_flow_to_trips_rows': None}

[]

OK - 5.1 build_flows happy path mínimo


### Smoke 5.2 - build_flows con segmentación y flow_to_trips

Qué prueba: integración básica de segmentación por mode y construcción del auxiliar flow_to_trips.

In [40]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")

options = FlowBuildOptions(
    h3_resolution=8,
    group_by=["mode"],
    time_aggregation="none",
    min_trips_per_flow=1,
    keep_flow_to_trips=True,
    require_validated=True,
)

flow_ds, report = build_flows(trips, options=options)

assert report.ok is True
assert "mode" in flow_ds.flows.columns
assert flow_ds.flow_to_trips is not None
assert set(flow_ds.flow_to_trips.columns) == {"flow_id", "movement_id"}

# Con los datos de fixture, deberían salir 3 flujos:
# - bus en OD principal
# - metro en OD principal
# - bus en OD secundario
assert len(flow_ds.flows) == 3
assert len(flow_ds.flow_to_trips) == 4

# Summary y parameters
assert report.summary["n_trips_in"] == 4
assert report.summary["n_flow_to_trips_rows"] == 4
assert report.parameters["group_by"] == ["mode"]

display(trips.data)
display(flow_ds.flows)
display(flow_ds.flows.sort_values(["origin_h3_index", "destination_h3_index", "mode"]).reset_index(drop=True))
display(flow_ds.flow_to_trips.sort_values(["flow_id", "movement_id"]).reset_index(drop=True))
show_ok("5.2 build_flows con segmentación y flow_to_trips")

,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


,flow_id,origin_h3_index,destination_h3_index,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,bus,2,3.0
1,flow_0000001,8828308281fffff,8828308285fffff,metro,1,1.5
2,flow_0000002,882830828dfffff,8828308287fffff,bus,1,3.0


,flow_id,origin_h3_index,destination_h3_index,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,bus,2,3.0
1,flow_0000001,8828308281fffff,8828308285fffff,metro,1,1.5
2,flow_0000002,882830828dfffff,8828308287fffff,bus,1,3.0


,flow_id,movement_id
0,flow_0000000,m0
1,flow_0000000,m1
2,flow_0000001,m2
3,flow_0000002,m3


OK - 5.2 build_flows con segmentación y flow_to_trips


### Smoke 5.3 - build_flows con agregación temporal horaria

Qué prueba: integración básica de bins temporales window_start_utc / window_end_utc.

In [46]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")

options = FlowBuildOptions(
    h3_resolution=8,
    group_by=["mode"],
    time_aggregation="hour",
    time_basis="origin",
    min_trips_per_flow=1,
    keep_flow_to_trips=False,
    require_validated=True,
)

flow_ds, report = build_flows(trips, options=options)

assert report.ok is True
assert "window_start_utc" in flow_ds.flows.columns
assert "window_end_utc" in flow_ds.flows.columns

# Con la fixture, deben salir 3 flujos horarios segmentados por mode
assert len(flow_ds.flows) == 3

# Validar que las ventanas sean consistentes
assert (flow_ds.flows["window_end_utc"] > flow_ds.flows["window_start_utc"]).all()

display(trips.data)
display(flow_ds.flows)
display(
    flow_ds.flows[
        ["flow_id", "origin_h3_index", "destination_h3_index", "mode", "window_start_utc", "window_end_utc", "flow_count", "flow_value"]
    ].sort_values(["window_start_utc", "mode"]).reset_index(drop=True)
)
display(report.summary)
show_ok("5.3 build_flows con agregación temporal horaria")

,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,2026-04-01 08:00:00,2026-04-01 09:00:00,bus,2,3.0
1,flow_0000001,8828308281fffff,8828308285fffff,2026-04-01 08:00:00,2026-04-01 09:00:00,metro,1,1.5
2,flow_0000002,882830828dfffff,8828308287fffff,2026-04-01 09:00:00,2026-04-01 10:00:00,bus,1,3.0


,flow_id,origin_h3_index,destination_h3_index,mode,window_start_utc,window_end_utc,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,bus,2026-04-01 08:00:00,2026-04-01 09:00:00,2,3.0
1,flow_0000001,8828308281fffff,8828308285fffff,metro,2026-04-01 08:00:00,2026-04-01 09:00:00,1,1.5
2,flow_0000002,882830828dfffff,8828308287fffff,bus,2026-04-01 09:00:00,2026-04-01 10:00:00,1,3.0


{'n_trips_in': 4,
 'n_trips_eligible': 4,
 'n_trips_dropped': 0,
 'n_flows_out': 3,
 'n_flow_to_trips_rows': None}

OK - 5.3 build_flows con agregación temporal horaria


### Smoke 5.4 - build_flows fatal por precondición de validación

Qué prueba: caso simple fatal cuando require_validated=True y el dataset de trips no está validado.

In [48]:
trips = make_tripdataset_for_flows(validated=False, tier="tier_1")

options = FlowBuildOptions(
    h3_resolution=8,
    require_validated=True,
)

try:
    _ = build_flows(trips, options=options)
    raise AssertionError("Se esperaba excepción por trips no validados.")
except ValidationError as error:
    print(error.issues)
    pass

show_ok("5.4 build_flows fatal por precondición de validación")

(Issue(level='error', code='FLOW.VALIDATION.REQUIRED_NOT_VALIDATED', message="build_flows requiere un TripDataset validado, pero metadata['is_validated']=False.", field=None, source_field=None, row_count=None, details={'check': 'precheck', 'require_validated': True, 'validated_flag': False, 'reason': 'validated_required', 'action': 'abort'}),)
OK - 5.4 build_flows fatal por precondición de validación


### Smoke 5.5 - export_flows happy path mínimo

Qué prueba: export mínimo desde un FlowDataset simple, sin extras, verificando artefactos y sidecar.

In [50]:
flows = make_flowdataset_for_export(with_extra_fields=False)
output_root = SMOKE_ROOT / "export_minimal_root"
output_root.mkdir(parents=True, exist_ok=True)

result, report = export_flows(
    flows,
    output_root=str(output_root),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="case_export_minimal",
        extra_flow_fields=None,
    ),
)

assert isinstance(result, FlowExportResult)
assert isinstance(report, OperationReport)
assert report.ok is True

assert set(result.artifacts.keys()) == {"flows", "locations", "metadata"}
assert Path(result.artifacts["flows"]).exists()
assert Path(result.artifacts["locations"]).exists()
assert Path(result.artifacts["metadata"]).exists()

flows_csv = pd.read_csv(result.artifacts["flows"])
locations_csv = pd.read_csv(result.artifacts["locations"])
metadata_json = json.loads(Path(result.artifacts["metadata"]).read_text(encoding="utf-8"))

assert list(flows_csv.columns) == ["origin", "dest", "count"]
assert {"id", "name", "lat", "lon"}.issubset(locations_csv.columns)
assert metadata_json["export"]["count_source"] == "flow_value"

display(flows.flows)
display(flows_csv)
display(locations_csv.head())
display(report.summary)
show_ok("5.5 export_flows happy path mínimo")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,2,3.0
1,flow_0000001,882830828dfffff,8828308287fffff,1,3.0


,origin,dest,count
0,8828308281fffff,8828308285fffff,3.0
1,882830828dfffff,8828308287fffff,3.0


,id,name,lat,lon
0,8828308281fffff,8828308281fffff,37.773515,-122.418271
1,882830828dfffff,882830828dfffff,37.767674,-122.410902
2,8828308285fffff,8828308285fffff,37.775910,-122.407876
3,8828308287fffff,8828308287fffff,37.781750,-122.415245


{'n_flows': 2,
 'n_locations': 4,
 'files_written': ['flows.csv', 'locations.csv', 'metadata.json']}

OK - 5.5 export_flows happy path mínimo


### Smoke 5.6 - export_flows con extras explícitas

Qué prueba: preservación selectiva de campos extra en flows.csv.

In [52]:
flows = make_flowdataset_for_export(with_extra_fields=True)
output_root = SMOKE_ROOT / "export_with_extras_root"
output_root.mkdir(parents=True, exist_ok=True)

result, report = export_flows(
    flows,
    output_root=str(output_root),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="case_export_with_extras",
        extra_flow_fields=["mode", "purpose", "window_start_utc"],
    ),
)

assert report.ok is True

flows_csv = pd.read_csv(result.artifacts["flows"])
metadata_json = json.loads(Path(result.artifacts["metadata"]).read_text(encoding="utf-8"))

assert {"origin", "dest", "count", "mode", "purpose", "window_start_utc"}.issubset(flows_csv.columns)
assert metadata_json["export"]["parameters"]["extra_flow_fields"] == ["mode", "purpose", "window_start_utc"]

display(flows.flows)
display(flows_csv)
show_ok("5.6 export_flows con extras explícitas")

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,window_start_utc,window_end_utc,mode,purpose,user_gender
0,flow_0000000,8828308281fffff,8828308285fffff,2,3.0,2026-04-01 08:00:00+00:00,2026-04-01 09:00:00+00:00,bus,work,M
1,flow_0000001,882830828dfffff,8828308287fffff,1,3.0,2026-04-01 09:00:00+00:00,2026-04-01 10:00:00+00:00,metro,study,F


,origin,dest,count,mode,purpose,window_start_utc
0,8828308281fffff,8828308285fffff,3.0,bus,work,2026-04-01 08:00:00+00:00
1,882830828dfffff,8828308287fffff,3.0,metro,study,2026-04-01 09:00:00+00:00


OK - 5.6 export_flows con extras explícitas


### Smoke 5.7 - export_flows fatal por colisión de directorio

Qué prueba: caso simple fatal cuando el directorio ya existe y mode="error_if_exists".

In [55]:
flows = make_flowdataset_for_export(with_extra_fields=False)
output_root = SMOKE_ROOT / "export_collision_root"
output_root.mkdir(parents=True, exist_ok=True)

# Primera exportación para crear el directorio
_ = export_flows(
    flows,
    output_root=str(output_root),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="case_collision",
    ),
)

# Segunda exportación al mismo directorio: debe fallar
try:
    _ = export_flows(
        flows,
        output_root=str(output_root),
        options=ExportFlowsOptions(
            format="flowmap_blue",
            mode="error_if_exists",
            folder_name="case_collision",
        ),
    )
    raise AssertionError("Se esperaba ExportError por colisión de directorio.")
except ExportError as error:
    print(error.issues)
    pass

show_ok("5.7 export_flows fatal por colisión de directorio")

(Issue(level='error', code='EXPORT_FLOWS.LAYOUT.EXPORT_DIR_EXISTS_ABORT', message="El directorio de exportación 'tmp_helper_tests\\\\smoke_tests\\\\export_collision_root\\\\case_collision' ya existe y mode='error_if_exists' impide continuar.", field=None, source_field=None, row_count=None, details={'output_root': 'tmp_helper_tests\\smoke_tests\\export_collision_root', 'export_dir': 'tmp_helper_tests\\smoke_tests\\export_collision_root\\case_collision', 'format': 'flowmap_blue', 'mode': 'error_if_exists', 'folder_name_effective': 'case_collision', 'artifact': 'export_dir', 'reason': 'export_dir_exists'}),)
OK - 5.7 export_flows fatal por colisión de directorio


### Smoke 5.8 - pipeline integrado build_flows -> export_flows

Qué prueba: integración básica de ambas funciones públicas en un caso pequeño pero representativo.

In [56]:
trips = make_tripdataset_for_flows(validated=True, tier="tier_1")

build_options = FlowBuildOptions(
    h3_resolution=8,
    group_by=["mode"],
    time_aggregation="hour",
    time_basis="origin",
    min_trips_per_flow=1,
    keep_flow_to_trips=False,
    require_validated=True,
)

flow_ds, build_report = build_flows(trips, options=build_options)

assert build_report.ok is True
assert flow_ds.metadata["events"][-1]["op"] == "build_flows"

output_root = SMOKE_ROOT / "integrated_build_export_root"
output_root.mkdir(parents=True, exist_ok=True)

export_result, export_report = export_flows(
    flow_ds,
    output_root=str(output_root),
    options=ExportFlowsOptions(
        format="flowmap_blue",
        mode="error_if_exists",
        folder_name="case_integrated_build_export",
        extra_flow_fields=["mode", "window_start_utc"],
    ),
)

assert export_report.ok is True
assert flow_ds.metadata["events"][-1]["op"] == "export_flows"

flows_csv = pd.read_csv(export_result.artifacts["flows"])
locations_csv = pd.read_csv(export_result.artifacts["locations"])
metadata_json = json.loads(Path(export_result.artifacts["metadata"]).read_text(encoding="utf-8"))

assert {"origin", "dest", "count", "mode", "window_start_utc"}.issubset(flows_csv.columns)
assert len(flows_csv) == build_report.summary["n_flows_out"]
assert metadata_json["flow_dataset_ref"]["aggregation_spec"]["group_by"] == ["mode"]
assert metadata_json["export"]["count_source"] == "flow_value"

print("Dataset de viajes")
display(trips.data)
print("\nExploracion de Flujos construidos")
display(flow_ds.flows)
display(build_report.issues)
display(flow_ds.metadata["events"])
print("\nExploracion de artefactos de exportación")
display(flows_csv)
display(locations_csv.head())
display(export_result)
display(export_report.issues)
display({"build_summary": build_report.summary, "export_summary": export_report.summary})
show_ok("5.8 pipeline integrado build_flows -> export_flows")

Dataset de viajes


,movement_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,user_gender,trip_weight
0,m0,8828308281fffff,8828308285fffff,2026-04-01 08:10:00+00:00,2026-04-01 08:35:00+00:00,bus,work,M,1.0
1,m1,8828308281fffff,8828308285fffff,2026-04-01 08:20:00+00:00,2026-04-01 08:50:00+00:00,bus,work,F,2.0
2,m2,8828308281fffff,8828308285fffff,2026-04-01 08:45:00+00:00,2026-04-01 09:05:00+00:00,metro,study,F,1.5
3,m3,882830828dfffff,8828308287fffff,2026-04-01 09:10:00+00:00,2026-04-01 09:40:00+00:00,bus,work,M,3.0



Exploracion de Flujos construidos


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,flow_count,flow_value
0,flow_0000000,8828308281fffff,8828308285fffff,2026-04-01 08:00:00,2026-04-01 09:00:00,bus,2,3.0
1,flow_0000001,8828308281fffff,8828308285fffff,2026-04-01 08:00:00,2026-04-01 09:00:00,metro,1,1.5
2,flow_0000002,882830828dfffff,8828308287fffff,2026-04-01 09:00:00,2026-04-01 10:00:00,bus,1,3.0


[]

...

,id,name,lat,lon
0,8828308281fffff,8828308281fffff,37.773515,-122.418271
1,882830828dfffff,882830828dfffff,37.767674,-122.410902
2,8828308285fffff,8828308285fffff,37.775910,-122.407876
3,8828308287fffff,8828308287fffff,37.781750,-122.415245


FlowExportResult(export_dir='tmp_helper_tests\\smoke_tests\\integrated_build_export_root\\case_integrated_build_export', artifacts={'flows': 'tmp_helper_tests\\smoke_tests\\integrated_build_export_root\\case_integrated_build_export\\flows.csv', 'locations': 'tmp_helper_tests\\smoke_tests\\integrated_build_export_root\\case_integrated_build_export\\locations.csv', 'metadata': 'tmp_helper_tests\\smoke_tests\\integrated_build_export_root\\case_integrated_build_export\\metadata.json'})

[]

{'build_summary': {'n_trips_in': 4,
  'n_trips_eligible': 4,
  'n_trips_dropped': 0,
  'n_flows_out': 3,
  'n_flow_to_trips_rows': None},
 'export_summary': {'n_flows': 3,
  'n_locations': 4,
  'files_written': ['flows.csv', 'locations.csv', 'metadata.json']}}

OK - 5.8 pipeline integrado build_flows -> export_flows
